# Suspended IDs

This notebook grabs additional data to flesh out suspect user IDs. PRAW is not available for suspended accounts, meaning we need to source data elsewhere. This process is different than the active account info collection because you can use Reddit.

In [ ]:
# ARCHIVED draft of the Arctic Shift history pull and reply collection (replaced by collection_scripts/tier4_arctic_full_history.py
# and p2_analysis/collect_replies_v4.py). Several cells below are unfinished experiments; two do not run.
# Libraries.
#import libraries
import pandas as pd
import numpy as np
import os 
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
from string import punctuation
import functools
import operator
from dotenv import load_dotenv
import praw
import prawcore
from prawcore.exceptions import NotFound
import pprint
import ast
import pprint
import os 
import itertools
import pickle
import requests
import time 
from requests_ratelimiter import LimiterSession
import pprint
import json
import base64
from itertools import islice
import pyarrow as pa
import pyarrow.parquet as pq
import random


In [ ]:
# Accounts from the first collection (User_Profiles.parquet).
users = pd.read_parquet('User_Profiles.parquet', engine='pyarrow')

In [ ]:
user_list=list(users['username'])

In [ ]:
u

In [ ]:
# 1,000 random accounts as a test set.
u = random.sample(user_list, 1000) 

In [ ]:
# Post history for each account (100 per request, 5 requests/second), skipping post ids already seen,
# flattened to one row per post and saved to postsample.pkl.
def get_historic_data(name_list):
    session = LimiterSession(per_second=5)
    user_submissions = {}
    seen_ids = set()  # track seen post ids across all users
    
    for name in name_list:
        print(name)
        all_posts = []
        after = None
        
        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/posts/search?author={name}&limit=100'
            if after:
                arctic_url += f'&after={after}'
            
            retries = 0
            success = False
            
            while retries < 10:
                print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)
                    
                    if response.status_code == 200:
                        api_return = response.__dict__
                        data_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))
                        
                        posts = data_content['data']
                        
                        # deduplicate during collection
                        for post in posts:
                            post_id = post.get('id')
                            if post_id not in seen_ids:
                                seen_ids.add(post_id)
                                all_posts.append(post)
                            else:
                                print(f'Duplicate post id found and skipped: {post_id}')
                        
                        if len(posts) < 100:
                            success = True
                            break
                        
                        after = posts[-1]['created_utc']
                        success = True
                        break
                    
                    else:
                        retries += 1
                        print(f'API response: {response.status_code}')
                
                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    retries += 1
            
            if not success or len(posts) < 100:
                break
        
        user_submissions[name] = all_posts
    
    # build DataFrame from all users
    rows = []
    for outer_key, records in user_submissions.items():
        for record in records:
            meta = record.get('_meta', {})
            row = {'outer_key': outer_key} | meta | {k: v for k, v in record.items() if k != '_meta'}
            rows.append(row)

    df = pd.DataFrame(rows)
    
    # deduplicate after building DataFrame as a safety net
    before = len(df)
    df = df.drop_duplicates(subset=['id'])
    after_dedup = len(df)
    if before != after_dedup:
        print(f'Dropped {before - after_dedup} duplicate rows from DataFrame')
    df = df.dropna(axis=1, how='all')
    df.to_pickle('postsample.pkl')
    #df.to_parquet('postsample.parquet', engine='pyarrow', compression='zstd')
    return df

In [ ]:
t=get_historic_data(u)

In [ ]:
t.author.nunique()

In [ ]:
# List the collected columns.
for col in t.columns:
    print (col)

In [ ]:
# 'Seed' pull: each account's posts (dir_name='posts') or comments, written to data/<dir_name>/ partitioned by account.
# Returns (account, item id) pairs used as seeds for the reply pull; failures go to data/failed_<dir_name>.csv.
def get_historic_seed(data_list, dir_name):
    session = LimiterSession(per_second=5)
    failed_users = []
    all_seed_tuples = []
    os.makedirs(f'data/{dir_name}', exist_ok=True)
    for d in data_list:
        F_start = time.perf_counter()
        print(d)
        all_data = []
        after = None
        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/{dir_name}/search?author={d}&limit=100'
            if after:
                arctic_url += f'&after={after}'
            retries = 0
            success = False
            data = []
            while retries < 10:
                try:
                    response = session.get(arctic_url)
                    if response.status_code == 200:
                        api_content = json.loads(
                            response.__dict__.get('_content').decode('utf-8')
                        )
                        data = api_content['data']
                        all_data.extend(data)
                        after = data[-1]['created_utc'] if len(data) == 100 else None
                        success = True
                        break
                    else:
                        retries += 1
                        print(f'API response: {response.status_code} for {d}')
                except Exception as e:
                    print(f'Error for {d}: {e.__class__}: {e}')
                    retries += 1
            if not success:
                print(f'Failed to collect data for {d} after 10 retries')
                failed_users.append({'user': d, 'reason': 'pagination failed'})
                break
            if len(data) < 100:
                F_end = time.perf_counter()
                print(f'User {d}: executed in {F_end - F_start:.4f} seconds')
                break
        if all_data:
            try:
                rows = []
                for record in all_data:
                    meta = record.get('_meta', {})
                    rows.append(
                        {'outer_key': d} | meta | {k: v for k, v in record.items() if k != '_meta'}
                    )
                    if record.get('id'):
                        all_seed_tuples.append((d, record.get('id')))
                df = pd.DataFrame(rows)
                for col in df.columns:
                    # explicitly handle edited column first since it can be
                    # a timestamp int, timestamp string, or False
                    if col == 'edited':
                        df[col] = df[col].apply(
                            lambda x: None if x is False else str(x)
                        )
                        continue
                    if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                        df[col] = df[col].apply(
                            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
                        )
                    types = df[col].dropna().apply(type).unique()
                    if len(types) > 1:
                        df[col] = df[col].apply(
                            lambda x: str(x) if not isinstance(x, (bool, type(None))) else x
                        )
                df = df.reset_index(drop=True)
                pq.write_to_dataset(
                    pa.Table.from_pandas(df, preserve_index=False),
                    root_path=f'data/{dir_name}',
                    partition_cols=['outer_key'],
                    compression='zstd'
                )
                print(f'Saved {len(rows)} records for {d}')
            except Exception as e:
                print(f'Failed to save data for {d}: {e.__class__}: {e}')
                failed_users.append({'user': d, 'reason': f'save error: {e}'})
        else:
            print(f'No data found for {d}')
    if failed_users:
        pd.DataFrame(failed_users).to_csv(f'data/failed_{dir_name}.csv', index=False)
        print(f'{len(failed_users)} users failed, saved to data/failed_{dir_name}.csv')
    print('Collection complete')
    return all_seed_tuples, dir_name

In [ ]:
get_historic_seed(u, 'posts')

In [ ]:
# Replies to each seed item (comments whose parent is the seed), saved in batches to data/<dir_name>_replies/.
def get_seed_replies(seed_tuple, dir_name, batch_size=100):
    dir_name_replies = f'{dir_name}_replies'
    session = LimiterSession(per_second=5)
    failed_seeds = []
    batch_rows = []

    os.makedirs(f'data/{dir_name_replies}', exist_ok=True)

    def save_batch(rows):
        if not rows:
            return
        df = pd.DataFrame(rows)
        for col in df.columns:
            if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                df[col] = df[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
                )
            types = df[col].dropna().apply(type).unique()
            if len(types) > 1:
                df[col] = df[col].apply(
                    lambda x: str(x) if not isinstance(x, (bool, type(None))) else x
                )
        df = df.reset_index(drop=True)
        pq.write_to_dataset(
            pa.Table.from_pandas(df, preserve_index=False),
            root_path=f'data/{dir_name_replies}',
            partition_cols=['outer_key'],
            compression='zstd'
        )
        print(f'Saved batch of {len(rows)} rows')

    for outer_key, seed_id in seed_tuple:
        all_data = []
        after = None

        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/comments/search?parent_id={seed_id}&limit=100&meta=true'
            if after:
                arctic_url += f'&after={after}'

            retries = 0
            success = False
            data = []

            while retries < 10:
                try:
                    response = session.get(arctic_url)

                    if response.status_code == 200:
                        api_content = json.loads(
                            response.__dict__.get('_content').decode('utf-8')
                        )
                        data = api_content['data']
                        all_data.extend(data)
                        after = data[-1]['created_utc'] if len(data) == 100 else None
                        success = True
                        break
                    else:
                        retries += 1
                        print(f'API response: {response.status_code} for {outer_key}')

                except Exception as e:
                    print(f'Error for {outer_key}: {e.__class__}: {e}')
                    retries += 1

            if not success:
                print(f'Failed to collect replies for {outer_key} after 10 retries')
                failed_seeds.append({'seed': outer_key, 'seed_id': seed_id, 'reason': 'pagination failed'})
                break

            if len(data) < 100:
                break

        # flatten and add to batch
        for record in all_data:
            meta = record.get('_meta', {})
            row = {'outer_key': outer_key, 'seed_id': seed_id} | meta | {k: v for k, v in record.items() if k != '_meta'}
            batch_rows.append(row)

        # save when batch is full
        if len(batch_rows) >= batch_size:
            save_batch(batch_rows)
            batch_rows = []  # reset batch

    # save any remaining rows
    if batch_rows:
        save_batch(batch_rows)

    if failed_seeds:
        pd.DataFrame(failed_seeds).to_csv(f'data/failed_{dir_name_replies}.csv', index=False)
        print(f'{len(failed_seeds)} seeds failed, saved to data/failed_{dir_name_replies}.csv')

    print('Collection complete')

In [ ]:
# Write a DataFrame to data/<dir_name>/ as parquet partitioned by account; nested values become JSON strings.
def save_to_parquet(df,dir_name):
    path=f'data/{dir_name}'
    os.makedirs(path, exist_ok=True)
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
            df[col] = df[col].apply(
                lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
            )
        if df[col].apply(lambda x: isinstance(x, (int, float, bool))).all():
            try:
                df[col] = df[col].astype(str)
            except:
                pass
                
    table = pa.Table.from_pandas(df)
    pq.write_to_dataset(
        table,
        root_path=path,
        partition_cols=['outer_key'],
        compression='zstd'
    )
    
    

In [ ]:
# Earlier version of the seed pull (kept for reference; it calls save_to_parquet before df exists).
def get_historic_seed(data_list,dir_name):
    
    session = LimiterSession(per_second=5)
    data_dict = {}
     
    for d in data_list:
        F_start = time.perf_counter()
        print(d)
        all_data = []
        after = None
        
        while True:
            F_start = time.perf_counter()
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/{dir_name}/search?author={d}&limit=100'
            if after:
                arctic_url += f'&after={after}'
            
            retries = 0
            success = False
            
            while retries < 10:
                #print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)
                    
                    if response.status_code == 200:
                        api_return = response.__dict__
                        api_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))
                        
                        data = api_content['data']
                        all_data.extend(data)
                        
                        if len(data) < 100:
                            success = True
                            break
                        
                        after = data[-1]['created_utc']
                        success = True
                        break
                    
                    else:
                        retries += 1
                        #print(f'API response: {response.status_code}')
                    
                
                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    raise e
                
            
            if not success or len(data) < 100:
                F_end = time.perf_counter()
                F_duration= F_end - F_start  
                print(f'User {d}: executed in {F_duration:.4f} seconds')
                break
                
        
        data_dict[d] = all_data
    
    # Build DataFrame from all users
        rows = []
        for outer_key, records in data_dict.items():
            for record in records:
                meta = record.get('_meta', {})
                row = {'outer_key': outer_key} | meta | {k: v for k, v in record.items() if k != '_meta'}
                rows.append(row)
                save_to_parquet(df,dir_name)

    df = pd.DataFrame(rows)
   
    seed_tuple=list(zip(df.outer_key, df.id))
    save_to_parquet(df,dir_name)
    
    return seed_tuple,dir_name,df

In [ ]:
s,d,f=get_historic_seed(u,'posts')

In [ ]:
# Earlier version of the reply pull that collects everything before saving.
def get_seed_replies(seed_tuple, dir_name):
    dir_name_replies = f'{dir_name}_replies'
    session = LimiterSession(per_second=5)
    data_dict = {}

    os.makedirs(f'data/{dir_name_replies}', exist_ok=True)

    for outer_key, seed_id in seed_tuple:
        all_data = []
        after = None

        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/comments/search?parent_id={seed_id}&limit=100'
            if after:
                arctic_url += f'&after={after}'

            retries = 0
            success = False
            
            while retries < 10:
                #print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)
                    
                    if response.status_code == 200:
                        api_return = response.__dict__
                        api_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))
                        
                        data = api_content['data']
                        all_data.extend(data)
                        
                        if len(data) < 100:
                            success = True
                            break
                        
                        after = data[-1]['created_utc']
                        success = True
                        break
                    
                    else:
                        retries += 1
                        #print(f'API response: {response.status_code}')
                    
                
                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    raise e
                
            
            if not success or len(data) < 100:
                  
                break
                
        
        data_dict[outer_key] = all_data
    
    # Build DataFrame from all users
    rows = []
    for outer_key, records in data_dict.items():
        for record in records:
            meta = record.get('_meta', {})
            row = {'outer_key': outer_key} | meta | {k: v for k, v in record.items() if k != '_meta'}
            rows.append(row)
            
    df = pd.DataFrame(rows)
    save_to_parquet(df,dir_name)
    return



In [ ]:
# Run the seed pull, then the reply pull, timing each.
def get_historic_data(data_list,dir_name):
    F1_start = time.perf_counter()
    seeds,dir_name=get_historic_seed(data_list,dir_name)
    F1_end = time.perf_counter()
    F1_duration=F1_end - F1_start
    print(f"Function one executed in {F1_duration:.4f} seconds")
    #rewrite store empties?

    F2_start = time.perf_counter()
    t=get_seed_replies(seeds,dir_name)
    F2_end = time.perf_counter()
    F2_duration=F2_end - F2_start
    print(f"Function two executed in {F2_duration:.4f} seconds")
    
    return t

In [ ]:

t=get_historic_data(u,'posts')

In [ ]:
t

In [ ]:
##

In [ ]:
# Unfinished draft: line 'data_dict[d]= [seed_id]all_data' is a syntax error, so this cell does not run.
def get_seed_replies(seed_dict,dir_name):
    dir_name=f'{dir_name}_replies'
    session = LimiterSession(per_second=5)
    data_dict = {}
     
    for d,seed_id in seed_dict.items():
        #print(d)
        all_data = []
        after = None
        
        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/comments/search?parent_id={seed_id}&limit=100'
            if after:
                arctic_url += f'&after={after}'
            
            retries = 0
            success = False
            
            while retries < 10:
                #print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)
                    
                    if response.status_code == 200:
                        api_return = response.__dict__
                        api_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))
                        
                        data = api_content['data']
                        all_data.extend(data)
                        
                        if len(data) < 100:
                            success = True
                            break
                        
                        after = data[-1]['created_utc']
                        success = True
                        break
                    
                    else:
                        retries += 1
                        #print(f'API response: {response.status_code}')
                
                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    raise e
            
            if not success or len(data) < 100:
                break
        
        data_dict[d]= [seed_id]all_data
    
    # Build DataFrame from all users
    rows = []
    for outer_key, seed_names in data_dict.items():
        for seed_names,records in data_dict.items():
            for record in records:
                meta = record.get('_meta', {})
                row = {'outer_key': outer_key} | meta | {k: v for k, v in record.items() if k != '_meta'}
                rows.append(row)

    df = pd.DataFrame(rows)
    save_to_parquet(df,dir_name)

    return df
    
    

In [ ]:
# Manual test: all comments in one thread (link_id) from Arctic Shift.
seed_id='z1c9z'
session = LimiterSession(per_second=5)
arctic_url = f'https://arctic-shift.photon-reddit.com/api/comments/search?link_id=z1c9z'
response = session.get(arctic_url)
api_return = response.__dict__
api_content = json.loads(api_return.get('_content').decode('utf-8'))

In [ ]:
api_content

In [ ]:
# Unfinished draft of a thread-reply pull; the cell is incomplete and does not run.
def get_historic_reply_data(origin_id_list):
    all_origin_replies = []
    for origin_id in origin_id_list:
        after_reply = None
        while True:
            reply_url = f'https://arctic-shift.photon-reddit.com/api/comments/search?link_id={origin_id}&limit=100&meta=true'
            if after_reply:
                reply_url += f'&after={after_reply}'
                
            retries = 0
            success = False
            replies = []

            while retries < 10:
                print(f'Retries are at: {retries}')
                try:
                    response = session.get(reply_url)
                    
                    if response.status_code == 200:
                        data_content = json.loads(
                            response.__dict__.get('_content').decode('utf-8')
                                    )
                        replies = data_content['data']
                        if len(replies) < 100:
                            success = True
                            break
                        
                        
                        for r in replies:
                            meta = r.get('_meta', {})
                            all_post_replies.append(
                                {'outer_key': d, 'reply_to_post_id': post_id} | meta | {k: v for k, v in r.items() if k != '_meta'}
                            )
                        
                
        
    

In [ ]:
pd.read_parquet('data.parquet', engine='pyarrow')